# Streaming Data That Processes ITSELF
### CONSUMER

> **A stream is a table to which rows keep being appended forever.**

You write the query as if it were a normal table. Spark takes care of:
- Detecting new files when they arrive in the bucket.
- Processing **only what is new** (micro-batch), not everything all over again.
- Saving progress in a **checkpoint** to recover from failures.

```
PRODUCER (other tab)             CONSUMER (this screen)
   drops files        ──────►    spark.readStream
   in the bucket                        │  transformations
                                        ▼
                                   writeStream → table / sink
```

---
## Limitations of Databricks Free / Community Edition

In the Free Edition, only **serverless compute** runs, which imposes streaming restrictions
that the book (written for classic clusters) does not mention:

- `trigger(processingTime="5 seconds")` → error `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`
- Continuous trigger by default → same error
- `trigger(availableNow=True)` → **the only supported trigger**: processes all available data and stops

**Implication:** instead of leaving the stream running continuously, the pattern is:
a file arrives in the bucket → we execute `process_new_data()` → it only processes what is new.
The concept taught (incremental, checkpoint, exactly-once) is **identical** to production;
the only thing that changes is who presses the button (you here, a scheduler in production).

---
## 0. Setup — Same Volume as the Producer

The paths **must match** the Producer notebook.

In [0]:
catalog = "main"
schema  = "streaming_live"
volume  = "raw_data"

base_path       = f"/Volumes/{catalog}/{schema}/{volume}"
source_path     = f"{base_path}/source"        # the "bucket" we monitor
checkpoint_path = f"{base_path}/checkpoint"    # stream progress status
output_table    = "sales_by_region"          # destination table (sink)

# Ensure the structure exists (in case this is run before the Producer)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
dbutils.fs.mkdirs(source_path)
spark.sql(f"USE {catalog}.{schema}")

print("Monitored bucket:", source_path)
print("Checkpoint      :", checkpoint_path)

Monitored bucket: /Volumes/main/streaming_live/raw_data/source
Checkpoint      : /Volumes/main/streaming_live/raw_data/checkpoint


In [0]:
from pyspark.sql.functions import col, sum as _sum, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define explicit schema (best practice in streaming to avoid inference)
order_schema = StructType([
    StructField("order_id",  StringType(),  True),
    StructField("product",  StringType(),  True),
    StructField("region",    StringType(),  True),
    StructField("quantity",  IntegerType(), True),
    StructField("price",    DoubleType(),  True),
    StructField("timestamp", StringType(),  True),
])
print("Schema defined.")

Schema defined.


---
## 1. `spark.readStream` — Subscribe to the Bucket

This **does not read anything yet**. It only registers that the source is a stream and returns a
*streaming DataFrame*: a plan built for data that might not have arrived yet.

`maxFilesPerTrigger=1` makes each micro-batch process one file at a time.

In [0]:
stream_df = (spark.readStream
    .schema(order_schema)
    .option("maxFilesPerTrigger", 1)
    .json(source_path))

print("Is it streaming?", stream_df.isStreaming)   # True
stream_df.printSchema()

Is it streaming? True
root
 |-- order_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- timestamp: string (nullable = true)



Notice: `isStreaming = True`. If we had used `spark.read.json(...)` it would be `False`.
This property is **inherited**: any transformation applied to `stream_df` becomes a stream operation as well.

---
## 2. Transformation — Same as in Batch

We calculate `total_sales` and aggregate sales by region.
**The exact same code you would use with static data** — that is the beauty of the structured model.

In [0]:
regional_summary = (stream_df
    .withColumn("total_sales", col("quantity") * col("price"))
    .groupBy("region")
    .agg(
        count("order_id").alias("total_orders"),
        _sum("total_sales").alias("total_sales"),
    ))

print("Is it still streaming?", regional_summary.isStreaming)  # True

Is it still streaming? True


---
## 3. `process_new_data()`

We encapsulate `writeStream` inside a function. Every time you run it:
1. It reads the files in the bucket that **haven't been processed yet** (thanks to the checkpoint).
2. It recalculates the summary and writes it to the destination table.
3. It stops (due to `availableNow`).

Since there is a `groupBy`, we use **`outputMode("complete")`**: it overwrites the entire summary table.

In [0]:
# Launches an incremental micro-batch: processes only new content from the bucket
def process_new_data():
    (regional_summary.writeStream
        .trigger(availableNow=True)
        .outputMode("complete")
        .option("checkpointLocation", checkpoint_path)
        .toTable(output_table)
        .awaitTermination())
    print("Incremental batch processed ✓")

print("Function ready. Execute process_new_data() when data arrives in the bucket.")

Function ready. Execute process_new_data() when data arrives in the bucket.


---
## 4. THE LIVE DEMO

1. In the other tab (Producer), run `generar_batch()` → a file drops into the bucket.
2. Here, execute `process_new_data()`.
3. Query the table → new data will appear.
4. Repeat: more data generated in the Producer → `process_new_data()` here → the table grows.

In [0]:
# STEP A: process whatever is currently in the bucket
process_new_data()

Incremental batch processed ✓


In [0]:
%sql
-- STEP B: view the results (this is a static table, no checkpoint required)
SELECT * FROM sales_by_region
ORDER BY total_sales DESC

region,total_orders,total_sales
west,7,10830.759999999998
north,3,7062.83
east,2,3220.76
south,2,2586.62
central,1,1209.3899999999999


Now request more data from the Producer and run the two cells above again.
You will see the summary update **without reprocessing old data**.

---
## 5. Highlight: Exactly-Once / Idempotence

Execute `process_new_data()` **twice in a row without generating new data**.
The table **does not change**: the checkpoint mechanism knows it already processed everything and skips the work.
This is the *exactly-once* data processing guarantee in action — each file is processed exactly one time.

In [0]:
# Without new data in the bucket: run twice and compare
process_new_data()
print("--- second execution without new data ---")
process_new_data()
print("The table remained identical. Spark reprocessed nothing.")

Incremental batch processed ✓
--- second execution without new data ---
Incremental batch processed ✓
The table remained identical. Spark reprocessed nothing.


---
## 6. Highlight: What is inside the checkpoint directory?

The checkpoint directory is the **memory** of the stream. If you delete it, the stream starts from scratch
and reprocesses everything. Let's look inside.

In [0]:
print("Checkpoint contents:\n")
for item in dbutils.fs.ls(checkpoint_path):
    data_type = "[DIR]" if item.isDir() else "     "
    print(f"  {data_type}  {item.name}")

print("""
  offsets/  -> which files in the bucket were reached by each micro-batch
  commits/  -> which micro-batches completed successfully
  state/    -> the state store (the aggregates from groupBy)
  metadata  -> general metadata about the stream
""")

Checkpoint contents:

  [DIR]  commits/
         metadata
  [DIR]  offsets/
  [DIR]  sources/
  [DIR]  state/

  offsets/  -> which files in the bucket were reached by each micro-batch
  commits/  -> which micro-batches completed successfully
  state/    -> the state store (the aggregates from groupBy)
  metadata  -> general metadata about the stream



---
## 7. How would this look in real PRODUCTION environments?

The only "sandbox-like" element here is that **we** are generating files and manually pushing the execution button.
In production, the producer is an external system, and the execution is triggered by an automated scheduler.
The **Spark code remains practically identical** — only the source layer changes. There are two primary architectural patterns:

---

### Family 1 — Files in Object Storage (closest to this demo)

Another system (an application, an export service, logs) drops files into an operational cloud storage bucket (S3 / GCS / ADLS).
The professional way to implement this is **not** raw `readStream.json`, but using Databricks **Auto Loader**:

```python
df = (spark.readStream
        .format("cloudFiles")                              # Auto Loader
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)  # infers and handles schema evolution
        .load("s3://my-bucket-real/landing/events/"))
```

Auto Loader provides functionalities that the basic `readStream.json` lacks: efficient file discovery at the scale of millions of files (via cloud notification services instead of directory listing), schema inference, schema evolution management, and rescuing corrupted records.

---

### Family 2 — Message Broker (true real-time streaming)

No intermediary storage files: a continuous stream of structured messages with end-to-end latency ranging from milliseconds to seconds.
This architecture is commonly deployed for fraud detection systems, IoT workloads, and clickstream tracking analysis:

```python
df = (spark.readStream
        .format("kafka")                                   # or kinesis / eventhubs / pubsub
        .option("kafka.bootstrap.servers", "broker1:9092,broker2:9092")
        .option("subscribe", "transactions")
        .load())
```

---

### The Stream Trigger in Production

In full-featured enterprise clusters (non-Free tiers), streaming jobs run as an **always-on daemon service**:
using `.trigger(processingTime="30 seconds")`, or set to `availableNow` orchestrations driven by an external **scheduled job workflow**.
Our manual execution of `process_new_data()` mimics exactly what that automated scheduler does independently.

---
## Clean Up

In [0]:
def clean_up():
    for q in spark.streams.active:
        print(f"Stopping: {q.name}")
        q.stop()
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.{output_table}")
    dbutils.fs.rm(base_path, True)
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
    print("Done ✓")

In [0]:
# Uncomment this to clear down and purge all metadata structures at the end of the session
#clean_up()

Done ✓
